# Enrichissement de `previous_application`

## Objectif

Ce notebook enrichira progressivement `previous_application.csv` avec les trois historiques liés par `SK_ID_PREV`. Cette première version traite uniquement `POS_CASH_balance.csv`, conformément à l'analyse et au choix de variables réalisés avant l'implémentation.

`POS_CASH_balance` contient plusieurs observations mensuelles pour une même ancienne demande. Elle doit donc être agrégée pour obtenir une ligne par `SK_ID_PREV` avant la jointure. La jointure gauche doit conserver les 1 670 214 anciennes demandes de `previous_application`. `installments_payments` et `credit_card_balance` seront étudiées séparément avant leur ajout.

In [1]:
from pathlib import Path

import pandas as pd

# ---------- Chemins des données ----------
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PREVIOUS_APPLICATION_PATH = RAW_DIR / "previous_application.csv"
POS_CASH_PATH = RAW_DIR / "POS_CASH_balance.csv"
PREVIOUS_ENRICHI_PATH = PROCESSED_DIR / "previous_application_enrichi.csv"

for data_path in [PREVIOUS_APPLICATION_PATH, POS_CASH_PATH]:
    if not data_path.is_file():
        raise FileNotFoundError(f"Fichier introuvable : {data_path.resolve()}")

## Chargement et contrôle initial

Des types compacts sont utilisés pour limiter la mémoire nécessaire au chargement des dix millions de lignes POS/CASH.

In [2]:
# ---------- Chargement des données ----------
previous_application = pd.read_csv(PREVIOUS_APPLICATION_PATH)
pos_cash = pd.read_csv(
    POS_CASH_PATH,
    dtype={
        "SK_ID_PREV": "int32",
        "SK_ID_CURR": "int32",
        "MONTHS_BALANCE": "int8",
        "SK_DPD": "int16",
        "SK_DPD_DEF": "int16",
        "NAME_CONTRACT_STATUS": "category",
    },
)

print(f"previous_application : {previous_application.shape}")
print(f"POS_CASH_balance : {pos_cash.shape}")
display(pos_cash.head())

previous_application : (1670214, 37)
POS_CASH_balance : (10001358, 8)


,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,1803195,182943,-31,48.0,45.0,Active,0,0
1,1715348,367990,-33,36.0,35.0,Active,0,0
2,1784872,397406,-32,12.0,9.0,Active,0,0
3,1903291,269225,-35,48.0,42.0,Active,0,0
4,2341044,334279,-35,36.0,35.0,Active,0,0


In [3]:
# ---------- Validation des données ----------
PREVIOUS_ID = "SK_ID_PREV"
MONTH_COLUMN = "MONTHS_BALANCE"
required_previous_columns = {PREVIOUS_ID, "SK_ID_CURR"}
required_pos_columns = {
    PREVIOUS_ID, "SK_ID_CURR", MONTH_COLUMN,
    "CNT_INSTALMENT_FUTURE", "SK_DPD_DEF",
}

if not required_previous_columns.issubset(previous_application.columns):
    missing = required_previous_columns.difference(previous_application.columns)
    raise KeyError(f"Colonnes absentes de previous_application : {sorted(missing)}")
if not required_pos_columns.issubset(pos_cash.columns):
    missing = required_pos_columns.difference(pos_cash.columns)
    raise KeyError(f"Colonnes absentes de POS_CASH_balance : {sorted(missing)}")

assert previous_application[PREVIOUS_ID].notna().all()
assert previous_application[PREVIOUS_ID].is_unique
assert pos_cash[[PREVIOUS_ID, "SK_ID_CURR", MONTH_COLUMN]].notna().all().all()
assert not pos_cash.duplicated([PREVIOUS_ID, MONTH_COLUMN]).any()
assert pos_cash.groupby(PREVIOUS_ID)["SK_ID_CURR"].nunique().le(1).all()
print("Validation des identifiants réussie.")

Validation des identifiants réussie.


## Logique de jointure et feature engineering entre `POS_CASH` et `previous_application`

### Colonnes transformées

- `MONTHS_BALANCE` produit `POS_MONTH_COUNT` et permet de sélectionner le mois le plus récent ;
- `CNT_INSTALMENT_FUTURE` produit `POS_REMAINING_INSTALLMENTS_LATEST` ;
- `SK_DPD_DEF` produit `POS_EVER_DPD` et `POS_EVER_SEVERE_DPD`.

`SK_DPD_DEF` est préféré à `SK_DPD`, car il ignore les petits impayés tolérés. Un retard sévère correspond ici à au moins 61 jours.

### Colonnes non retenues

- `SK_ID_CURR` est redondante avec celle de la table parente et sert uniquement au contrôle de cohérence ;
- `CNT_INSTALMENT` est susceptible d'évoluer et apporte une information proche des échéances restantes ;
- `NAME_CONTRACT_STATUS` possède plusieurs catégories mensuelles non prioritaires ;
- `SK_DPD` est remplacée par sa version avec tolérance.

Les colonnes brutes ne sont jamais supprimées de leur fichier source.

In [4]:
# ---------- Feature engineering ----------
pos_cash = pos_cash.assign(
    _IS_DPD=pos_cash["SK_DPD_DEF"].gt(0).astype("int8"),
    _IS_SEVERE_DPD=pos_cash["SK_DPD_DEF"].ge(61).astype("int8"),
)

latest_row_index = pos_cash.groupby(PREVIOUS_ID)[MONTH_COLUMN].idxmax()
latest_installments = (
    pos_cash.loc[latest_row_index, [PREVIOUS_ID, "CNT_INSTALMENT_FUTURE"]]
    .set_index(PREVIOUS_ID)
    .rename(columns={
        "CNT_INSTALMENT_FUTURE": "POS_REMAINING_INSTALLMENTS_LATEST"
    })
)

pos_cash_agg = (
    pos_cash.groupby(PREVIOUS_ID, observed=True)
    .agg(
        POS_MONTH_COUNT=(MONTH_COLUMN, "size"),
        POS_EVER_DPD=("_IS_DPD", "max"),
        POS_EVER_SEVERE_DPD=("_IS_SEVERE_DPD", "max"),
    )
    .join(latest_installments, validate="one_to_one")
    .reset_index()
)

assert pos_cash_agg[PREVIOUS_ID].is_unique
del pos_cash
print(f"POS_CASH agrégé : {pos_cash_agg.shape}")
display(pos_cash_agg.head())

POS_CASH agrégé : (936325, 5)


,SK_ID_PREV,POS_MONTH_COUNT,POS_EVER_DPD,POS_EVER_SEVERE_DPD,POS_REMAINING_INSTALLMENTS_LATEST
0,1000001,3,0,0,0.0
1,1000002,5,0,0,0.0
2,1000003,4,0,0,9.0
3,1000004,8,0,0,0.0
4,1000005,11,0,0,0.0


In [5]:
# ---------- Couverture des clés ----------
previous_ids = pd.Index(previous_application[PREVIOUS_ID])
pos_ids = pd.Index(pos_cash_agg[PREVIOUS_ID])
orphan_pos_ids = pos_ids.difference(previous_ids)
previous_without_pos_ids = previous_ids.difference(pos_ids)

print(f"Identifiants POS absents de previous_application : {len(orphan_pos_ids):,}")
print(f"Anciennes demandes sans historique POS : {len(previous_without_pos_ids):,}")

Identifiants POS absents de previous_application : 37,422
Anciennes demandes sans historique POS : 771,311


## réalisation de la jointure entre `POS_CASH` et `previous_application`

La jointure gauche conserve toutes les anciennes demandes. Les indicateurs de retard restent manquants en l'absence d'historique, tandis que `POS_HAS_HISTORY` permet d'identifier explicitement cette situation.

In [6]:
# ---------- Jointure des données ----------
previous_application_enrichi = previous_application.merge(
    pos_cash_agg, on=PREVIOUS_ID, how="left", validate="one_to_one"
)
previous_application_enrichi["POS_HAS_HISTORY"] = (
    previous_application_enrichi["POS_MONTH_COUNT"].notna().astype("int8")
)
previous_application_enrichi["POS_MONTH_COUNT"] = (
    previous_application_enrichi["POS_MONTH_COUNT"].fillna(0).astype("int16")
)

assert len(previous_application_enrichi) == len(previous_application)
assert previous_application_enrichi[PREVIOUS_ID].is_unique

In [7]:
# ---------- Contrôle avant et après la jointure ----------
controle_jointure = pd.DataFrame({
    "étape": ["avant", "après"],
    "nombre de lignes": [len(previous_application), len(previous_application_enrichi)],
    "nombre de colonnes": [
        previous_application.shape[1], previous_application_enrichi.shape[1]
    ],
})
display(controle_jointure)
print("Toutes les validations sont réussies.")

,étape,nombre de lignes,nombre de colonnes
0,avant,1670214,37
1,après,1670214,42


Toutes les validations sont réussies.


In [8]:
# ---------- Aperçu après l'étape POS/CASH ----------
display(previous_application_enrichi.head())

,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,...,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL,POS_MONTH_COUNT,POS_EVER_DPD,POS_EVER_SEVERE_DPD,POS_REMAINING_INSTALLMENTS_LATEST,POS_HAS_HISTORY
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,17145.0,SATURDAY,15,...,-42.0,300.0,-42.0,-37.0,0.0,2,0.0,0.0,0.0,1
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,NaN,607500.0,THURSDAY,11,...,-134.0,916.0,365243.0,365243.0,1.0,5,0.0,0.0,32.0,1
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,NaN,112500.0,TUESDAY,11,...,-271.0,59.0,365243.0,365243.0,1.0,10,0.0,0.0,3.0,1
3,2819243,176158,Cash loans,47041.335,450000.0,470790.0,NaN,450000.0,MONDAY,7,...,-482.0,-152.0,-182.0,-177.0,1.0,12,0.0,0.0,0.0,1
4,1784265,202054,Cash loans,31924.395,337500.0,404055.0,NaN,337500.0,THURSDAY,9,...,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0


## Logique de jointure et feature engineering entre `installments_payments` et `previous_application`


Une échéance peut être répartie sur plusieurs lignes de paiement. Le traitement comporte donc deux niveaux : consolidation par échéance, puis agrégation par `SK_ID_PREV`. Cette méthode évite de compter plusieurs fois le montant attendu.

### Traitement retenu

La clé logique d'une échéance est composée de `SK_ID_PREV`, `NUM_INSTALMENT_NUMBER` et `NUM_INSTALMENT_VERSION`. Pour chaque échéance, la date et le montant attendus sont uniques, les paiements partiels sont additionnés et leur date la plus tardive est retenue.

Les variables finales sont : nombre d'échéances, proportion de retards, retard maximal, proportion de sous-paiements et rapport entre total payé et total attendu. `SK_ID_CURR` sert uniquement au contrôle de cohérence, car la jointure utilise `SK_ID_PREV`. Les paiements sans date ou montant réel sont exclus des ratios concernés au lieu d'être assimilés à zéro.

In [9]:
# ---------- Chargement des paiements ----------
INSTALLMENTS_PATH = RAW_DIR / "installments_payments.csv"
if not INSTALLMENTS_PATH.is_file():
    raise FileNotFoundError(f"Fichier introuvable : {INSTALLMENTS_PATH.resolve()}")

installments = pd.read_csv(
    INSTALLMENTS_PATH,
    dtype={
        "SK_ID_PREV": "int32",
        "SK_ID_CURR": "int32",
        "NUM_INSTALMENT_VERSION": "float32",
        "NUM_INSTALMENT_NUMBER": "int16",
        "DAYS_INSTALMENT": "float32",
        "DAYS_ENTRY_PAYMENT": "float32",
        "AMT_INSTALMENT": "float32",
        "AMT_PAYMENT": "float32",
    },
)
print(f"installments_payments : {installments.shape}")
display(installments.head())

installments_payments : (13605401, 8)


,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,1054186,161674,1.0,6,-1180.0,-1187.0,6948.359863,6948.359863
1,1330831,151639,0.0,34,-2156.0,-2156.0,1716.525024,1716.525024
2,2085231,193053,2.0,1,-63.0,-63.0,25425.000000,25425.000000
3,2452527,199697,1.0,3,-2418.0,-2426.0,24350.130859,24350.130859
4,2714724,167756,1.0,2,-1383.0,-1366.0,2165.040039,2160.584961


In [10]:
# ---------- Validation des paiements ----------
INSTALLMENT_KEYS = [
    PREVIOUS_ID, "NUM_INSTALMENT_NUMBER", "NUM_INSTALMENT_VERSION"
]
required_installment_columns = set(INSTALLMENT_KEYS).union({
    "SK_ID_CURR", "DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT",
    "AMT_INSTALMENT", "AMT_PAYMENT",
})
if not required_installment_columns.issubset(installments.columns):
    missing = required_installment_columns.difference(installments.columns)
    raise KeyError(f"Colonnes absentes de installments_payments : {sorted(missing)}")

assert installments[INSTALLMENT_KEYS + ["SK_ID_CURR"]].notna().all().all()
assert installments.groupby(PREVIOUS_ID)["SK_ID_CURR"].nunique().le(1).all()
print(
    "Lignes supplémentaires partageant une clé d'échéance : "
    f"{installments.duplicated(INSTALLMENT_KEYS).sum():,}"
)

Lignes supplémentaires partageant une clé d'échéance : 653,483


In [11]:
# ---------- Consolidation par échéance ----------
installments["_HAS_PAYMENT"] = installments["AMT_PAYMENT"].notna().astype("int8")

installment_level = (
    installments.groupby(INSTALLMENT_KEYS, as_index=False, observed=True)
    .agg(
        INSTALLMENT_DUE_DATE=("DAYS_INSTALMENT", "first"),
        INSTALLMENT_LAST_PAYMENT_DATE=("DAYS_ENTRY_PAYMENT", "max"),
        INSTALLMENT_EXPECTED_AMOUNT=("AMT_INSTALMENT", "first"),
        INSTALLMENT_PAID_AMOUNT=("AMT_PAYMENT", "sum"),
        _HAS_PAYMENT=("_HAS_PAYMENT", "max"),
    )
)
installment_level["INSTALLMENT_PAID_AMOUNT"] = (
    installment_level["INSTALLMENT_PAID_AMOUNT"]
    .where(installment_level["_HAS_PAYMENT"].eq(1))
)

assert not installment_level.duplicated(INSTALLMENT_KEYS).any()
del installments
print(f"Échéances consolidées : {installment_level.shape[0]:,}")

Échéances consolidées : 12,951,918


In [12]:
# ---------- Feature engineering des échéances ----------
observed_date = installment_level["INSTALLMENT_LAST_PAYMENT_DATE"].notna()
observed_amount = installment_level["INSTALLMENT_PAID_AMOUNT"].notna()
days_late = (
    installment_level["INSTALLMENT_LAST_PAYMENT_DATE"]
    - installment_level["INSTALLMENT_DUE_DATE"]
).clip(lower=0)

installment_level["_DAYS_LATE"] = days_late.where(observed_date)
installment_level["_IS_LATE"] = (
    days_late.gt(0).astype("float32").where(observed_date)
)
installment_level["_IS_UNDERPAID"] = (
    installment_level["INSTALLMENT_PAID_AMOUNT"]
    .lt(installment_level["INSTALLMENT_EXPECTED_AMOUNT"] - 0.01)
    .astype("float32")
    .where(observed_amount)
)
installment_level["_EXPECTED_AMOUNT_OBSERVED"] = (
    installment_level["INSTALLMENT_EXPECTED_AMOUNT"].where(observed_amount)
)

In [13]:
# ---------- Agrégation par ancienne demande ----------
installment_grouped = installment_level.groupby(PREVIOUS_ID, observed=True)
installments_agg = installment_grouped.agg(
    INSTAL_INSTALLMENT_COUNT=(PREVIOUS_ID, "size"),
    INSTAL_LATE_PAYMENT_RATIO=("_IS_LATE", "mean"),
    INSTAL_MAX_DAYS_LATE=("_DAYS_LATE", "max"),
    INSTAL_UNDERPAYMENT_RATIO=("_IS_UNDERPAID", "mean"),
)
payment_totals = installment_grouped[
    ["INSTALLMENT_PAID_AMOUNT", "_EXPECTED_AMOUNT_OBSERVED"]
].sum(min_count=1)
expected_denominator = payment_totals["_EXPECTED_AMOUNT_OBSERVED"].mask(
    payment_totals["_EXPECTED_AMOUNT_OBSERVED"].eq(0)
)
installments_agg["INSTAL_PAYMENT_RATIO"] = (
    payment_totals["INSTALLMENT_PAID_AMOUNT"] / expected_denominator
)
installments_agg = installments_agg.reset_index()

assert installments_agg[PREVIOUS_ID].is_unique
del installment_level, installment_grouped, payment_totals
print(f"Paiements agrégés : {installments_agg.shape}")
display(installments_agg.head())

Paiements agrégés : (997752, 6)


,SK_ID_PREV,INSTAL_INSTALLMENT_COUNT,INSTAL_LATE_PAYMENT_RATIO,INSTAL_MAX_DAYS_LATE,INSTAL_UNDERPAYMENT_RATIO,INSTAL_PAYMENT_RATIO
0,1000001,2,0.0,0.0,0.0,1.0
1,1000002,4,0.0,0.0,0.0,1.0
2,1000003,3,0.0,0.0,0.0,1.0
3,1000004,7,0.0,0.0,0.0,1.0
4,1000005,10,0.2,3.0,0.0,1.0


## Réalisation de la jointure entre `installments_payments` et `previous_application`

Les caractéristiques de paiement sont désormais agrégées avec une ligne par `SK_ID_PREV`. Elles peuvent donc être jointes à `previous_application` sans multiplier les anciennes demandes. Une jointure gauche conserve toutes les demandes, y compris celles sans historique de paiement.

In [14]:
# ---------- Couverture et jointure ----------
installment_ids = pd.Index(installments_agg[PREVIOUS_ID])
previous_enriched_ids = pd.Index(previous_application_enrichi[PREVIOUS_ID])
print(
    "Identifiants de paiements absents de previous_application : "
    f"{len(installment_ids.difference(previous_enriched_ids)):,}"
)
print(
    "Anciennes demandes sans historique de paiement : "
    f"{len(previous_enriched_ids.difference(installment_ids)):,}"
)

rows_before_installment_join = len(previous_application_enrichi)
columns_before_installment_join = previous_application_enrichi.shape[1]
previous_application_enrichi = previous_application_enrichi.merge(
    installments_agg, on=PREVIOUS_ID, how="left", validate="one_to_one"
)
previous_application_enrichi["INSTAL_HAS_HISTORY"] = (
    previous_application_enrichi["INSTAL_INSTALLMENT_COUNT"]
    .notna().astype("int8")
)
previous_application_enrichi["INSTAL_INSTALLMENT_COUNT"] = (
    previous_application_enrichi["INSTAL_INSTALLMENT_COUNT"]
    .fillna(0).astype("int16")
)

assert len(previous_application_enrichi) == rows_before_installment_join
assert previous_application_enrichi[PREVIOUS_ID].is_unique

Identifiants de paiements absents de previous_application : 38,847
Anciennes demandes sans historique de paiement : 711,309


In [15]:
# ---------- Contrôle avant et après la jointure ----------
controle_installments = pd.DataFrame({
    "étape": ["avant installments", "après installments"],
    "nombre de lignes": [rows_before_installment_join, len(previous_application_enrichi)],
    "nombre de colonnes": [
        columns_before_installment_join, previous_application_enrichi.shape[1]
    ],
})
display(controle_installments)
print("Validation de la jointure installments réussie.")

,étape,nombre de lignes,nombre de colonnes
0,avant installments,1670214,42
1,après installments,1670214,48


Validation de la jointure installments réussie.


In [16]:
# ---------- Export et contrôle du fichier ----------
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
previous_application_enrichi.to_csv(PREVIOUS_ENRICHI_PATH, index=False)
export_head = pd.read_csv(PREVIOUS_ENRICHI_PATH, nrows=5)
print(f"Table exportée vers : {PREVIOUS_ENRICHI_PATH.resolve()}")
display(export_head)

Table exportée vers : C:\Users\Philippe MAGNE\Documents\3 - DEV\P6_initiez_vous_au_MLOps_(partie1_sur_2)\data\processed\previous_application_enrichi.csv


,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,...,POS_EVER_DPD,POS_EVER_SEVERE_DPD,POS_REMAINING_INSTALLMENTS_LATEST,POS_HAS_HISTORY,INSTAL_INSTALLMENT_COUNT,INSTAL_LATE_PAYMENT_RATIO,INSTAL_MAX_DAYS_LATE,INSTAL_UNDERPAYMENT_RATIO,INSTAL_PAYMENT_RATIO,INSTAL_HAS_HISTORY
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,17145.0,SATURDAY,15,...,0.0,0.0,0.0,1,1,0.000000,0.0,0.0,1.0,1
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,NaN,607500.0,THURSDAY,11,...,0.0,0.0,32.0,1,5,0.000000,0.0,0.0,1.0,1
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,NaN,112500.0,TUESDAY,11,...,0.0,0.0,3.0,1,9,0.111111,1.0,0.0,1.0,1
3,2819243,176158,Cash loans,47041.335,450000.0,470790.0,NaN,450000.0,MONDAY,7,...,0.0,0.0,0.0,1,11,0.000000,0.0,0.0,1.0,1
4,1784265,202054,Cash loans,31924.395,337500.0,404055.0,NaN,337500.0,THURSDAY,9,...,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,0


## Conclusion de l'étape `installments_payments`

Les paiements multiples ont été consolidés au niveau de l'échéance, puis résumés par `SK_ID_PREV`. La jointure ajoute uniquement les caractéristiques validées et conserve une ligne par ancienne demande. `credit_card_balance` sera analysée séparément avant toute nouvelle modification.